This notebook tunes STL order weights for the reward function

In [1]:
%load_ext autoreload
%autoreload 2

import os
import time
from collections import defaultdict

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from IPython.display import display
from joblib import Parallel, delayed

from race_env import RaceEnv
from stl import STLOrderMonitor, STL_REWARDS

In [2]:
COURSE_PATH = "../../ros2_ws/src/pylon_2026_world/purt_course.yaml"
# inherited settings
SETTINGS = {"MAX_STEPS": 400, "NOISE": 0.25, "DISCOUNT_FACTOR": 0.99, "ALPHA": 0.1, "EPSILON": 0.3,
            "test_rollouts": 200, "episodes": 3000}

# parallel training
N_JOBS = max(1, (os.cpu_count() or 2) - 2)

In [4]:
# configs
cell_size = 1.0
first_stage = 1 
stl_order_weights = list(range(0, 11)) # 0-10

# p_Gk is positive when we stay inside r1 and the opposite
# ρ_¬Gk+1 is positive when we stay outside r2 and the opposite
# h > 0: either we get closer to gate k or farther to gate k+1
# theta_2 = 0
# theta_1 = r1 = r2 = 3.0 (do later tuning)
# now tune the order weights

r1, r2 = 3.0, 9.5
seeds = [67, 123, 42] # every weight is trained once per seed; seeds[0] is also used for the baseline
theta_1, theta_2 = -r1, 0.0 
h_bins = [theta_1, theta_2]

In [5]:
class VisitCounter(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.visited = set()

    def reset(self):
        obs, info = self.env.reset()
        self.visited.add(obs)
        return obs, info

    def step(self, action, ignore_truncation=False):
        out = self.env.step(action, ignore_truncation)
        self.visited.add(out[0])
        return out

def greedy_action(Q, state, n_actions):
    return int(np.argmax(Q[state])) if state in Q else np.random.randint(n_actions)
    
def evaluate_greedy(env, Q, n_rollouts):
    lap_steps, rho_order, rho_avoidance = [], [], []
    for _ in range(n_rollouts):
        state, _ = env.reset()
        while True:
            state, _, done, truncated, info = env.step(greedy_action(Q, state, env.action_space.n))
            if done or truncated:
                break
        if info["lap_finished"]:
            lap_steps.append(len(env.reward_history))
            rho_order.append(info["rho_order"])
            rho_avoidance.append(info["rho_avoidance"])
    finished = bool(lap_steps)
    return {"lap_rate": len(lap_steps) / n_rollouts,
            "lap_steps": float(np.mean(lap_steps)) if finished else np.nan,
            "rho_order": float(np.mean(rho_order)) if finished else np.nan,
            "rho_order_min": float(np.min(rho_order)) if finished else np.nan,
            "rho_avoidance": float(np.mean(rho_avoidance)) if finished else np.nan,
            "rho_avoidance_min": float(np.min(rho_avoidance)) if finished else np.nan}

def q_learning(env, discount_factor, alpha, epsilon, n_episodes):
    n_actions = env.action_space.n
    Q = defaultdict(lambda: np.random.random(n_actions))
    for _ in range(n_episodes):
        state, _ = env.reset()
        while True:
            action = np.random.randint(n_actions) if np.random.random() < epsilon else int(np.argmax(Q[state]))
            next_state, reward, done, truncated, _ = env.step(action)
            Q[state][action] += alpha * (reward + discount_factor * np.max(Q[next_state]) * (1 - done) - Q[state][action])
            state = next_state
            if done or truncated:
                break
    return Q

def train(stl_order, seed=seeds[0]):
    np.random.seed(seed)
    env = RaceEnv(COURSE_PATH, CELL_SIZE_M=cell_size, MAX_STEPS=SETTINGS["MAX_STEPS"], NOISE=SETTINGS["NOISE"])
    # stage 0's ρ_gate shaping hurts training when stl_order > 0; at stl_order = 0 there is no order reward, so include it
    stages_from = 0 if stl_order == 0 else first_stage
    monitor = STLOrderMonitor(env, REWARDS={**STL_REWARDS, "stl_order": stl_order}, r1=r1, r2=r2, first_stage=stages_from, H_BINS=h_bins)
    visits = VisitCounter(monitor)
    start = time.perf_counter()
    Q = q_learning(visits, SETTINGS["DISCOUNT_FACTOR"], SETTINGS["ALPHA"], SETTINGS["EPSILON"], SETTINGS["episodes"])
    return monitor, Q, len(visits.visited), time.perf_counter() - start

def train_one(stl_order, seed):
    monitor, Q, states, train_s = train(stl_order, seed)
    return {"stl_order": stl_order, "seed": seed, "train_s": train_s, "states": states,
            **evaluate_greedy(monitor, Q, SETTINGS["test_rollouts"])}

def stage_distances(env, Q, n_rollouts):
    rows = []
    for ep in range(n_rollouts):
        state, _ = env.reset()
        d_next = {}
        while True:
            q = env.q
            _, nxt = env.targets()
            state, _, done, truncated, info = env.step(greedy_action(Q, state, env.action_space.n))
            if nxt is not None:
                d_next[q] = min(d_next.get(q, np.inf), env.gate_distance(info["new_xy"], nxt))
            if done or truncated:
                break
        if info["lap_finished"]:
            rows += [{"rollout": ep, "stage": q, "d_next": d} for q, d in d_next.items()]
    return pd.DataFrame(rows)

**Note 1:**
- p_Gk = r1 - d_G_k
- p_Gk+1 = d_G_k+1 - r_2 (positive when outside r2, negative when inside r2)

-> r_2 needs to be > min(d_G_k+1) to make p_Gk+1 (give both negative and postivie rewards)

**Note 2:**
- The distance from the first pylon is only 7.6m, which is smaller than our chosen r_2, which means W will always be negative when we fly from gate 0-1.
- Because the W in from G1-2 and 2-3 is mostly positive, the minimum of W will always be from stage 0 and can never be recovered
-  Reward: the order reward is weight × ΔH. In stage 0, ΔH comes from the start geometry, so it's just noise in the reward. A larger weight means more noise, which hurts training.

-> I dropped stage 0 during using our stl order robustness score

In [6]:
# find the min distance to gate k+1 in stages (1, 2)
baseline, Q_base, _, _ = train(0)
base_d = stage_distances(baseline, Q_base, SETTINGS["test_rollouts"])
base_d = base_d[base_d["stage"] >= first_stage] # w=0 run includes stage 0; keep the r2 analysis on the tuned stages

G = baseline.env.GATES
n = len(G)
geometry = []
for k in range(first_stage, n - 1): 
    nxt = k + 1
    mid = 0.5 * (G[k]["p1"] + G[k]["p2"])
    geometry.append({"stage": k, "Target": G[k]["name"], "Avoid": G[nxt]["name"],
                     "d(mid target, avoid) (m)": baseline.gate_distance(mid, nxt)})
stage_table = (pd.DataFrame(geometry)
               .merge(base_d.groupby("stage")["d_next"].agg(["mean", "min"]).reset_index(), on="stage", how="left")
               .rename(columns={"mean": "Baseline mean min d (m)", "min": "Baseline worst min d (m)"}))
display(stage_table.style.format(precision=2, na_rep="-").hide(axis="index"))

print(f"Baseline lap rate {base_d['rollout'].nunique() / SETTINGS['test_rollouts']:.0%}; "
      f"smallest baseline min d over all stages = {base_d['d_next'].min():.2f} m")

stage,Target,Avoid,"d(mid target, avoid) (m)",Baseline mean min d (m),Baseline worst min d (m)
1,G2,G3,7.91,10.73,9.33
2,G3,G4,7.86,11.47,9.83


Baseline lap rate 100%; smallest baseline min d over all stages = 9.33 m


In [7]:
# weights tuning
jobs = [(w, s) for w in stl_order_weights for s in seeds]
print(f"Training {len(jobs)} runs ({len(stl_order_weights)} STL order weights x seeds {seeds}) on {N_JOBS} workers")

runs = []
start = time.perf_counter()
for r in Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(delayed(train_one)(w, s) for w, s in jobs):
    runs.append(r)
    print(f"[{len(runs):>2}/{len(jobs)}] {time.perf_counter() - start:>6.0f}s  stl_order {r['stl_order']:<3} seed {r['seed']:<3}: "
          f"lap {r['lap_rate']:.1%}, {r['lap_steps']:.1f} steps, "
          f"ρ_order {r['rho_order']:.2f} m, ρ_avoid {r['rho_avoidance']:.2f} m", flush=True)
print(f"Training took {time.perf_counter() - start:.0f}s wall ({sum(r['train_s'] for r in runs):.0f}s of training)")

Training 33 runs (11 STL order weights x seeds [67, 123, 42]) on 20 workers
[ 1/33]    383s  stl_order 6   seed 67 : lap 100.0%, 71.5 steps, ρ_order 0.66 m, ρ_avoid 0.27 m
[ 2/33]    401s  stl_order 5   seed 123: lap 100.0%, 73.8 steps, ρ_order 1.24 m, ρ_avoid 0.18 m
[ 3/33]    414s  stl_order 3   seed 42 : lap 100.0%, 75.8 steps, ρ_order 1.14 m, ρ_avoid 0.26 m
[ 4/33]    416s  stl_order 1   seed 67 : lap 100.0%, 70.7 steps, ρ_order 1.45 m, ρ_avoid 0.67 m
[ 5/33]    416s  stl_order 1   seed 123: lap 100.0%, 77.4 steps, ρ_order 1.73 m, ρ_avoid 0.24 m
[ 6/33]    420s  stl_order 4   seed 67 : lap 100.0%, 72.8 steps, ρ_order 1.17 m, ρ_avoid 0.24 m
[ 7/33]    422s  stl_order 6   seed 123: lap 100.0%, 77.6 steps, ρ_order 1.59 m, ρ_avoid 0.29 m
[ 8/33]    433s  stl_order 0   seed 42 : lap 100.0%, 72.6 steps, ρ_order -1.55 m, ρ_avoid 0.51 m
[ 9/33]    434s  stl_order 2   seed 42 : lap 100.0%, 90.2 steps, ρ_order 1.59 m, ρ_avoid 0.30 m
[10/33]    438s  stl_order 2   seed 67 : lap 100.0%, 75.6 s

In [20]:
lap_col = f"Lap @{SETTINGS['episodes']}"

per_seed = pd.DataFrame(runs).sort_values(["stl_order", "seed"]).reset_index(drop=True)

results = (per_seed.groupby("stl_order")
           .agg(lap_rate=("lap_rate", "mean"),
                lap_steps=("lap_steps", "mean"), lap_steps_std=("lap_steps", "std"),
                rho_order=("rho_order", "mean"), rho_order_std=("rho_order", "std"),
                rho_order_min=("rho_order_min", "min"),
                rho_avoidance=("rho_avoidance", "mean"), rho_avoidance_min=("rho_avoidance_min", "min"),
                states=("states", "mean"), train_s=("train_s", "mean"))
           .reset_index()
           .rename(columns={"stl_order": "STL weight", "lap_rate": lap_col,
                            "lap_steps": "Steps/lap", "lap_steps_std": "± steps",
                            "rho_order": "ρ_order (m)", "rho_order_std": "± ρ_order",
                            "rho_order_min": "Worst ρ_order (m)",
                            "rho_avoidance": "ρ_avoid (m)", "rho_avoidance_min": "Worst ρ_avoid (m)",
                            "states": "States", "train_s": "Train s"}))

print(f"Mean ± std over seeds {seeds}")
display(results.style
        .format({lap_col: "{:.1%}", "Steps/lap": "{:.1f}", "± steps": "{:.1f}",
                 "ρ_order (m)": "{:.2f}", "± ρ_order": "{:.2f}", "Worst ρ_order (m)": "{:.2f}",
                 "ρ_avoid (m)": "{:.2f}", "Worst ρ_avoid (m)": "{:.2f}", "States": "{:.0f}", "Train s": "{:.0f}"},
                na_rep="-")
        .background_gradient(subset=[lap_col], cmap="magma", vmin=0, vmax=1)
        .background_gradient(subset=["Steps/lap"], cmap="magma_r")
        .background_gradient(subset=["ρ_order (m)"], cmap="magma")
        .hide(axis="index"))

Mean ± std over seeds [67, 123, 42]


STL weight,Lap @3000,Steps/lap,± steps,ρ_order (m),± ρ_order,Worst ρ_order (m),ρ_avoid (m),Worst ρ_avoid (m),States,Train s
0,99.8%,79.3,7.5,-2.29,0.64,-2.81,0.39,-1.37,6290,416
1,100.0%,75.4,4.1,1.62,0.15,0.73,0.49,-0.43,5668,402
2,100.0%,83.5,7.4,1.65,0.08,0.48,0.26,-1.35,5277,419
3,100.0%,79.4,3.1,1.61,0.43,0.68,0.23,-0.55,4951,414
4,100.0%,74.8,1.7,1.32,0.19,0.30,0.24,-1.32,4741,415
5,100.0%,77.6,5.0,1.51,0.24,0.57,0.24,-0.80,4732,405
6,100.0%,76.9,5.0,1.17,0.47,0.20,0.28,-0.50,4797,343
7,100.0%,78.2,5.0,1.25,0.57,0.30,0.21,-1.35,4772,252
8,68.0%,134.3,91.4,1.33,0.35,0.95,0.21,-0.55,4652,252
9,100.0%,80.3,7.9,1.11,0.02,0.20,0.23,-0.63,4805,244
